In [1]:
%%capture
%pip install pycox lifelines scikit-survival optuna tqdm

In [2]:
import gc
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torchtuples as tt
import xgboost as xgb
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from pycox.models import CoxPH
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxnetSurvivalAnalysis
from tqdm.auto import tqdm
from sksurv.metrics import integrated_brier_score
from pycox.models import CoxTime
from pycox.preprocessing.label_transforms import LabTransCoxTime
from pycox.evaluation import EvalSurv
from pycox.models.cox_time import MLPVanillaCoxTime
from sklearn.model_selection import cross_val_score

In [ ]:
import warnings

# Suppress the specific PyTorch indexing warning caused by torchtuples
warnings.filterwarnings(
    action="ignore",
    message=".*non-tuple sequence for multidimensional indexing.*"
)

In [4]:
# utilise gpu
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

Training on: cuda


In [5]:
np.random.seed(1)
n_samples = 6000
d = 10

# Linear model
X = np.random.uniform(-1, 1, (n_samples, d)).astype("float32")
# log-risk function
f_x = X[:, 0] + 2*X[:, 1]

lambda_0 = 1.0
U = np.random.uniform(0, 1, n_samples)
# survival times: T = - log(U) / (lambda_0 * exp(f(x)))
T = -np.log(U) / (lambda_0 * np.exp(f_x))

# censoring 50% similar to paper
C = np.random.exponential(scale=np.mean(T), size=n_samples)

time = np.minimum(T, C).astype("float32")  # choose minimum
event = (T <= C).astype("float32")  # event indicator

In [6]:
# quadratic non-linear model
alpha = 3.0 # Controls the steepness of the "U" shape
X_nl_quad = np.random.uniform(-1, 1, (n_samples, d)).astype("float32")

# f(x) = alpha * (x0^2 + x1^2)
# Risk is lowest at (0,0) and highest at the corners/edges
f_x_nl_quad = alpha * (X_nl_quad[:, 0]**2 + X_nl_quad[:, 1]**2)

U_quad = np.random.uniform(0, 1, n_samples)

# survival times: T = - log(U) / (lambda_0 * exp(f(x)))
T_nl_quad = -np.log(U_quad) / (lambda_0 * np.exp(f_x_nl_quad))

# censoring ~50%
C_nl_quad = np.random.exponential(scale=np.mean(T_nl_quad), size=n_samples)

time_nl_quad = np.minimum(T_nl_quad, C_nl_quad).astype("float32")
event_nl_quad = (T_nl_quad <= C_nl_quad).astype("float32")

In [7]:
from sksurv.datasets import load_flchain

# FLCHAIN data
df_flchain, y_flchain = load_flchain()

# The "chapter" column contains many missing values, so it is standard practice to drop it
df_flchain = df_flchain.drop("chapter", axis=1)

# Drop any remaining rows with NaNs to ensure clean training
clean_indices = df_flchain.dropna().index
df_flchain_clean = df_flchain.loc[clean_indices]
y_flchain_clean = y_flchain[clean_indices]

# Convert categorical variables
df_flchain_numeric = pd.get_dummies(df_flchain_clean, drop_first=True)

# Extract X, T, E
X_flchain = df_flchain_numeric.astype("float32")
T_flchain = y_flchain_clean["futime"].astype("float32")
E_flchain = y_flchain_clean["death"].astype("float32")

In [8]:
from sksurv.datasets import load_gbsg2

# GBSG data
df_gbsg, y_gbsg = load_gbsg2()
df_gbsg_numeric = pd.get_dummies(df_gbsg, drop_first=True)

X_gbsg = df_gbsg_numeric.astype("float32")
T_gbsg = y_gbsg["time"].astype("float32")
E_gbsg = y_gbsg["cens"].astype("float32")

In [35]:
from pycox.datasets import support

# Load SUPPORT data
df_support = support.read_df()

# Separate features from the target variables
X_raw = df_support.drop(columns=["duration", "event"])
y_support = df_support[["duration", "event"]]

# One-hot encode categorical variables
df_support_numeric = pd.get_dummies(X_raw, drop_first=True)

# Cast to float32
X_support = df_support_numeric.astype("float32").values
T_support = y_support["duration"].astype("float32").values
E_support = y_support["event"].astype("float32").values

In [9]:
# cox proportional hazards
def cph(X_train, T_train, E_train, X_test, times):
    df_train = pd.DataFrame(X_train)
    df_train["time"] = T_train
    df_train["event"] = E_train

    cph = CoxPHFitter(penalizer=0.0)  # no penalisation

    try:
        cph.fit(df_train, duration_col="time", event_col="event")

        # predictions flattened to a 1D array
        preds = cph.predict_partial_hazard(pd.DataFrame(X_test)).values.flatten()

        # return 2d  array for IBS
        surv = cph.predict_survival_function(pd.DataFrame(X_test), times=times).values.T

        return preds, surv

    except Exception as e:
        print(f"CPH failed to converge: {e}")
        return None, None

In [10]:
def cox_lasso(X_train, T_train, E_train, X_test, times=None):
    try:
        # scikit-survival requires a structured array for the target variable
        y_train = np.array([(bool(e), t) for e, t in zip(E_train, T_train)],
                           dtype=[('event', 'bool'), ('time', 'float32')])

        # Initialize Lasso (l1_ratio=1.0 is pure Lasso / L1 penalty)
        lasso_model = CoxnetSurvivalAnalysis(l1_ratio=1.0, fit_baseline_model=True)

        # fit model
        lasso_model.fit(X_train, y_train)

        # predict and flatten to ensure a clean 1D array for the bootstrap loop
        risk_scores = lasso_model.predict(X_test).flatten()

        # 2d array of survival functions for IBS
        surv_funcs = lasso_model.predict_survival_function(X_test)
        surv = np.array([fn(times) for fn in surv_funcs])

        return risk_scores, surv

    except Exception as e:
        print(f"Cox Lasso failed to converge: {e}")
        return None, None

In [11]:
# random survival forest
def rsf(X_train, T_train, E_train, X_test, params, times=None):
    try:
        y_train = np.empty(len(E_train), dtype=[("e", bool), ("t", float)])
        y_train["e"] = E_train.astype(bool)
        y_train["t"] = T_train
        # Apply parameters dynamically, falling back to original defaults if keys are missing
        rsf_model = RandomSurvivalForest(
            n_estimators=params.get("n_estimators", 100),
            min_samples_split=params.get("min_samples_split", 2),
            min_samples_leaf=params.get("min_samples_leaf", 15),
            max_features=params.get("max_features", "sqrt"),
            n_jobs=-1,
            random_state=1
        )
        rsf_model.fit(X_train, y_train)

        # risk for c-index
        risk = rsf_model.predict(X_test)

        # survival functions for IBS
        surv_funcs = rsf_model.predict_survival_function(X_test)

        if times is not None:
            surv = np.array([fn(times) for fn in surv_funcs]).astype(np.float32)
        else:
            surv = None

        return risk, surv

    except Exception as e:
        print(f"RSF failed: {e}")
        return None, None

In [12]:
def optimise_rsf(X_tr, T_tr, E_tr, X_val, T_val, E_val, n_trials=20):
    # Format the split target variables into the structured array required by sksurv
    y_tr = np.array(
        [(bool(e), float(t)) for e, t in zip(E_tr, T_tr)],
        dtype=[("e", bool), ("t", float)]
    )
    y_val = np.array(
        [(bool(e), float(t)) for e, t in zip(E_val, T_val)],
        dtype=[("e", bool), ("t", float)]
    )

    def objective(trial):
        # Define the search space
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 20, 50, step=10),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 3, 20),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None])
        }

        # Initialise and fit on the 80% sub-training set
        rsf = RandomSurvivalForest(**params, n_jobs=-1, random_state=1)
        rsf.fit(X_tr, y_tr)

        # Evaluate c-index on the 20% validation set
        c_index = rsf.score(X_val, y_val)

        # Memory cleanup between trials
        del rsf
        gc.collect()

        return c_index

    # Run Optuna Study
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize")

    study.optimize(objective, n_trials=n_trials, n_jobs=-1, show_progress_bar=True)

    # Return only the parameters
    return study.best_params

In [13]:
def deepsurv(X_train, T_train, E_train, X_test, params, times):
    try:
        # Ensure arrays are float32 for PyTorch compatibility
        X_train = X_train.astype("float32")
        T_train = T_train.astype("float32")
        E_train = E_train.astype("float32")
        X_test = X_test.astype("float32")

        # Split strictly within the training set for Early Stopping
        X_tr, X_val, T_tr, T_val, E_tr, E_val = train_test_split(
            X_train, T_train, E_train, test_size=0.2, random_state=1, stratify=E_train
        )

        y_tr = (T_tr, E_tr)
        y_val = (T_val, E_val)

        # Network features
        in_features = X_tr.shape[1]
        out_features = 1
        num_nodes = [params["nodes"]] * params["layers"]
        dropout = params["dropout"]
        batch_norm = params.get("batch_norm", True)

        # Activation function
        activation = nn.SELU if params.get("activation", "ReLU") == "SELU" else nn.ReLU

        net = tt.practical.MLPVanilla(
            in_features, num_nodes, out_features, batch_norm,
            dropout, activation=activation, output_bias=False
        )

        if params.get("optimiser", "adam") == "adam":
            optim = tt.optim.Adam(lr=params["lr"], weight_decay=params.get("l2", 0.0))
        elif params.get("optimiser") == "sgd":
            optim = tt.optim.SGD(lr=params["lr"], momentum=params.get("momentum", 0.9),
                                 weight_decay=params.get("l2", 0.0), nesterov=True)
        else:
            raise ValueError("Unknown optimiser")

        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = CoxPH(net, optim, device=device)

        callbacks = [tt.callbacks.EarlyStopping(patience=15)]
        batch_size = params.get("batch_size", 64)


        # Fit model
        model.fit(
            X_tr, y_tr,
            batch_size=batch_size,
            epochs=500,
            callbacks=callbacks,
            verbose=False,
            val_data=(X_val, y_val)
        )

        # Return the flat array of log-hazard (risk) scores
        risk = model.predict_net(X_test).flatten()

        # survival probabilities
        _ = model.compute_baseline_hazards(X_tr, y_tr)  # calculate baseline hazards
        surv_df = model.predict_surv_df(X_test)
        surv = surv_df.reindex(times, method='ffill').bfill().values.T

        return risk, surv

    except Exception as e:
        print(f"DeepSurv failed: {e}")
        return None

In [14]:
def optimise_deepsurv(X_tr, T_tr, E_tr, X_val, T_val, E_val, n_trials=20):
    def objective(trial):
        # parameters to optimise
        n_nodes = trial.suggest_int("nodes", 16, 256, step=16)
        n_layers = trial.suggest_int("layers", 1, 4)
        dropout = trial.suggest_float("dropout", 0.1, 0.6)
        lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
        l2 = trial.suggest_float("l2", 1e-4, 1e-1, log=True)
        batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128, 256])
        activation_name = trial.suggest_categorical("activation", ["ReLU", "SELU"])
        batch_norm = trial.suggest_categorical("batch_norm", [True, False])

        in_features = X_tr.shape[1]
        out_features = 1
        num_nodes = [n_nodes] * n_layers
        activation = nn.SELU if activation_name == "SELU" else nn.ReLU

        # deepsurv
        net = tt.practical.MLPVanilla(
            in_features, num_nodes, out_features,
            batch_norm=batch_norm, dropout=dropout,
            activation=activation, output_bias=False
        )

        device = "cuda" if torch.cuda.is_available() else "cpu"
        optim = tt.optim.Adam(lr=lr, weight_decay=l2)
        model = CoxPH(net, optim, device=device)

        callbacks = [tt.callbacks.EarlyStopping(patience=15)]

        model.fit(
            X_tr, (T_tr, E_tr),
            batch_size=batch_size, epochs=500,
            callbacks=callbacks, verbose=False,
            val_data=(X_val, (T_val, E_val))
        )

        # PyCox native evaluation
        _ = model.compute_baseline_hazards()
        surv = model.predict_surv_df(X_val)
        ev = EvalSurv(surv, T_val, E_val, censor_surv="km")
        c_index = ev.concordance_td()

        # memory cleanup
        del model, net, optim
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        return c_index

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=True)

    return study.best_params

In [36]:
# Train/test split

# Sim Linear
X_train_lin, X_test_lin, T_train_lin, T_test_lin, E_train_lin, E_test_lin = train_test_split(
    X, time, event, test_size=0.2, random_state=42, stratify=event
)

# Sim Nonlinear Quadratic
X_train_quad, X_test_quad, T_train_quad, T_test_quad, E_train_quad, E_test_quad = train_test_split(
    X_nl_quad, time_nl_quad, event_nl_quad, test_size=0.2, random_state=42, stratify=event_nl_quad
)

# gbsg
X_train_gbsg, X_test_gbsg, T_train_gbsg, T_test_gbsg, E_train_gbsg, E_test_gbsg = train_test_split(
    X_gbsg, T_gbsg, E_gbsg, test_size=0.2, random_state=42, stratify=E_gbsg
)

# flchain
X_train_fl, X_test_fl, T_train_fl, T_test_fl, E_train_fl, E_test_fl = train_test_split(
    X_flchain, T_flchain, E_flchain, test_size=0.2, random_state=42, stratify=E_flchain
)

# support
X_train_sup, X_test_sup, T_train_sup, T_test_sup, E_train_sup, E_test_sup = train_test_split(
    X_support, T_support, E_support, test_size=0.2, random_state=42, stratify=E_support
)

In [37]:
# Scaling features

# Sim Linear
scaler_lin = StandardScaler()
X_train_lin_scaled = scaler_lin.fit_transform(X_train_lin)
X_test_lin_scaled = scaler_lin.transform(X_test_lin).astype("float32")

# Sim Nonlinear Quadratic
scaler_quad = StandardScaler()
X_train_quad_scaled = scaler_quad.fit_transform(X_train_quad)
X_test_quad_scaled = scaler_quad.transform(X_test_quad).astype("float32")

# GBSG
scaler_gbsg = StandardScaler()
X_train_gbsg_scaled = scaler_gbsg.fit_transform(X_train_gbsg)
X_test_gbsg_scaled = scaler_gbsg.transform(X_test_gbsg).astype("float32")

# FLCHAIN
scaler_fl = StandardScaler()
X_train_fl_scaled = scaler_fl.fit_transform(X_train_fl)
X_test_fl_scaled = scaler_fl.transform(X_test_fl).astype("float32")

# SUPPORT
scaler_sup = StandardScaler()
X_train_sup_scaled = scaler_sup.fit_transform(X_train_sup)
X_test_sup_scaled = scaler_sup.transform(X_test_sup).astype("float32")

In [38]:
# Validation split for hyperparameter tuning

# Sim Linear
X_tr_lin, X_val_lin, T_tr_lin, T_val_lin, E_tr_lin, E_val_lin = train_test_split(
    X_train_lin_scaled, T_train_lin, E_train_lin, test_size=0.2, random_state=1, stratify=E_train_lin
)

# Sim Nonlinear Quadratic
X_tr_quad, X_val_quad, T_tr_quad, T_val_quad, E_tr_quad, E_val_quad = train_test_split(
    X_train_quad_scaled, T_train_quad, E_train_quad, test_size=0.2, random_state=1, stratify=E_train_quad
)

# GBSG
X_tr_gbsg, X_val_gbsg, T_tr_gbsg, T_val_gbsg, E_tr_gbsg, E_val_gbsg = train_test_split(
    X_train_gbsg_scaled, T_train_gbsg, E_train_gbsg, test_size=0.2, random_state=1, stratify=E_train_gbsg
)

# FLCHAIN
X_tr_fl, X_val_fl, T_tr_fl, T_val_fl, E_tr_fl, E_val_fl = train_test_split(
    X_train_fl_scaled, T_train_fl, E_train_fl, test_size=0.2, random_state=1, stratify=E_train_fl
)

# SUPPORT
X_tr_sup, X_val_sup, T_tr_sup, T_val_sup, E_tr_sup, E_val_sup = train_test_split(
    X_train_sup_scaled, T_train_sup, E_train_sup, test_size=0.2, random_state=1, stratify=E_train_sup
)

In [32]:
n_trials = 20

# best_linear_rsf = optimise_rsf(X_tr_lin, T_tr_lin, E_tr_lin, X_val_lin, T_val_lin, E_val_lin, n_trials=n_trials)
# params_linear_rsf = best_linear_rsf.copy()

# best_nonlinear_quad_rsf = optimise_rsf(X_tr_quad, T_tr_quad, E_tr_quad, X_val_quad, T_val_quad, E_val_quad, n_trials=n_trials)
# params_nonlinear_quad_rsf = best_nonlinear_quad_rsf.copy()


# # gbsg
# best_gbsg_params_rsf = optimise_rsf(X_tr_gbsg, T_tr_gbsg, E_tr_gbsg, X_val_gbsg, T_val_gbsg, E_val_gbsg, n_trials=n_trials)
# params_gbsg_rsf = best_gbsg_params_rsf.copy()

# # flchain
# best_fl_params_rsf = optimise_rsf(X_tr_fl, T_tr_fl, E_tr_fl, X_val_fl, T_val_fl, E_val_fl, n_trials=n_trials)
# params_flchain_rsf = best_fl_params_rsf.copy()

# support
best_sup_params_rsf = optimise_rsf(X_tr_sup, T_tr_sup, E_tr_sup, X_val_sup, T_val_sup, E_val_sup, n_trials=n_trials)
params_sup_rsf = best_sup_params_rsf.copy()

  0%|          | 0/20 [00:00<?, ?it/s]

In [42]:
print(f"params_linear_rsf = {params_linear_rsf}")
print(f"params_nonlinear_quad_rsf = {params_nonlinear_quad_rsf}")
print(f"params_gbsg_rsf = {params_gbsg_rsf}")
print(f"params_flchain_rsf = {params_flchain_rsf}")
print(f"params_sup_rsf = {params_sup_rsf}")

params_linear_rsf = {'n_estimators': 30, 'min_samples_split': 6, 'min_samples_leaf': 17, 'max_features': None}
params_nonlinear_quad_rsf = {'n_estimators': 30, 'min_samples_split': 10, 'min_samples_leaf': 18, 'max_features': None}
params_gbsg_rsf = {'n_estimators': 30, 'min_samples_split': 11, 'min_samples_leaf': 17, 'max_features': 'log2'}
params_flchain_rsf = {'n_estimators': 30, 'min_samples_split': 8, 'min_samples_leaf': 14, 'max_features': 'sqrt'}
params_sup_rsf = {'n_estimators': 40, 'min_samples_split': 5, 'min_samples_leaf': 6, 'max_features': None}


In [ ]:
params_linear_rsf = {'n_estimators': 30, 'min_samples_split': 6, 'min_samples_leaf': 17, 'max_features': None}
params_nonlinear_quad_rsf = {'n_estimators': 30, 'min_samples_split': 10, 'min_samples_leaf': 18, 'max_features': None}
params_gbsg_rsf = {'n_estimators': 30, 'min_samples_split': 11, 'min_samples_leaf': 17, 'max_features': 'log2'}
params_flchain_rsf = {'n_estimators': 30, 'min_samples_split': 8, 'min_samples_leaf': 14, 'max_features': 'sqrt'}

In [39]:
n_trials = 20

# best_linear = optimise_deepsurv(X_tr_lin, T_tr_lin, E_tr_lin, X_val_lin, T_val_lin, E_val_lin, n_trials=n_trials)
# params_linear = best_linear.copy()

# best_nonlinear_quad = optimise_deepsurv(X_tr_quad, T_tr_quad, E_tr_quad, X_val_quad, T_val_quad, E_val_quad, n_trials=n_trials)
# params_nonlinear_quad = best_nonlinear_quad.copy()


# # gbsg
# best_gbsg_params = optimise_deepsurv(X_tr_gbsg, T_tr_gbsg, E_tr_gbsg, X_val_gbsg, T_val_gbsg, E_val_gbsg, n_trials=n_trials)
# params_gbsg = best_gbsg_params.copy()

# # flchain
# best_fl_params = optimise_deepsurv(X_tr_fl, T_tr_fl, E_tr_fl, X_val_fl, T_val_fl, E_val_fl, n_trials=n_trials)
# params_flchain = best_fl_params.copy()

# support
best_sup_params = optimise_deepsurv(X_tr_sup, T_tr_sup, E_tr_sup, X_val_sup, T_val_sup, E_val_sup, n_trials=n_trials)
params_sup = best_sup_params.copy()

  0%|          | 0/20 [00:00<?, ?it/s]

In [40]:
# Print all optimal hyperparameters
print(f"params_linear = {params_linear}")
print(f"params_nonlinear_quad = {params_nonlinear_quad}")
print(f"params_gbsg = {params_gbsg}")
print(f"params_flchain = {params_flchain}")
print(f"params_sup = {params_sup}")

params_linear = {'nodes': 176, 'layers': 1, 'dropout': 0.34272986330985167, 'lr': 0.003843462763409894, 'l2': 0.010007979565186667, 'batch_size': 64, 'activation': 'SELU', 'batch_norm': False}
params_nonlinear_quad = {'nodes': 80, 'layers': 1, 'dropout': 0.35726812096086596, 'lr': 0.0008270597937631612, 'l2': 0.020180349515626226, 'batch_size': 32, 'activation': 'ReLU', 'batch_norm': True}
params_gbsg = {'nodes': 96, 'layers': 3, 'dropout': 0.3138593042789217, 'lr': 0.009094446103670615, 'l2': 0.08534239584290268, 'batch_size': 32, 'activation': 'ReLU', 'batch_norm': True}
params_flchain = {'nodes': 112, 'layers': 3, 'dropout': 0.20205939164189893, 'lr': 0.00016452363842751923, 'l2': 0.0043586599582238535, 'batch_size': 128, 'activation': 'ReLU', 'batch_norm': True}
params_sup = {'nodes': 224, 'layers': 2, 'dropout': 0.4717771694003404, 'lr': 0.00044718400419894907, 'l2': 0.0016303735888695827, 'batch_size': 16, 'activation': 'ReLU', 'batch_norm': False}


In [22]:
def coxtime(X_train, T_train, E_train, X_test, params, times):
    try:
        labtrans = LabTransCoxTime()
        y_train_cox = labtrans.fit_transform(T_train, E_train)

        X_tr, X_val, y_tr_time, y_val_time, y_tr_event, y_val_event = train_test_split(
            X_train, y_train_cox[0], y_train_cox[1],
            test_size=0.2, random_state=1, stratify=y_train_cox[1]
        )

        y_tr = (y_tr_time, y_tr_event)
        y_val = (y_val_time, y_val_event)

        in_features = X_train.shape[1]
        num_nodes_list = [params.get("nodes", 32)] * params.get("layers", 2)
        dropout = params.get("dropout", 0.2)

        # Use MLPVanillaCoxTime which handles extra time covariate
        net = MLPVanillaCoxTime(
            in_features=in_features,
            num_nodes=num_nodes_list,
            batch_norm=True,
            dropout=dropout
        )

        device = "cuda" if torch.cuda.is_available() else "cpu"

        # Use canonical cox-time model
        model = CoxTime(net, tt.optim.Adam, labtrans=labtrans, device=device)
        model.optimizer.set_lr(params.get("lr", 1e-3))

        callbacks = [tt.callbacks.EarlyStopping(patience=10)]

        # Train model
        model.fit(
            X_tr, y_tr,
            batch_size=params.get("batch_size", 256),
            epochs=200,
            callbacks=callbacks,
            verbose=False,
            val_data=(X_val, y_val)
        )

        # Predict
        model.compute_baseline_hazards()
        surv_df = model.predict_surv_df(X_test)

        # Compute risk (using negative expected survival as a proxy for Cox-Time)
        expected_survival_time = surv_df.sum().values
        risk = -expected_survival_time

        # Compute survival probabilities at requested times
        surv = surv_df.reindex(times, method="ffill").bfill().values.T

        return risk, surv

    except Exception as e:
        print(f"Cox-Time evaluation failed: {e}")
        return None, None

In [23]:
def optimise_coxtime(X_tr, T_tr, E_tr, X_val, T_val, E_val, n_trials=15):
    # Ensure float32 for PyTorch/PyCox compatibility
    X_tr = X_tr.astype("float32")
    T_tr = T_tr.astype("float32")
    E_tr = E_tr.astype("float32")
    X_val = X_val.astype("float32")
    T_val = T_val.astype("float32")
    E_val = E_val.astype("float32")

    # Transform the target variables for Cox-Time outside the loop for speed.
    labtrans = LabTransCoxTime()
    y_tr_cox = labtrans.fit_transform(T_tr, E_tr)
    y_val_cox = labtrans.transform(T_val, E_val)

    y_tr = (y_tr_cox[0], y_tr_cox[1])
    y_val = (y_val_cox[0], y_val_cox[1])

    def objective(trial):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Parameters to optimise
        n_nodes = trial.suggest_int("nodes", 32, 256, step=32)
        n_layers = trial.suggest_int("layers", 1, 4)
        dropout = trial.suggest_float("dropout", 0.1, 0.5)
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [64, 128, 256, 512])

        # Build the model architecture using CoxTime MLP
        in_features = X_tr.shape[1]
        num_nodes_list = [n_nodes] * n_layers

        net = MLPVanillaCoxTime(
            in_features=in_features,
            num_nodes=num_nodes_list,
            batch_norm=True,
            dropout=dropout
        )

        device = "cuda" if torch.cuda.is_available() else "cpu"

        # Canonical optimiser setup
        model = CoxTime(net, tt.optim.Adam, labtrans=labtrans, device=device)
        model.optimizer.set_lr(lr)
        callbacks = [tt.callbacks.EarlyStopping(patience=10)]

        # Train the model
        model.fit(
            X_tr, y_tr,
            batch_size=batch_size, epochs=200,
            callbacks=callbacks, verbose=False,
            val_data=(X_val, y_val)
        )

        # Compute baseline hazards
        model.compute_baseline_hazards()

        # Predict the full survival dataframe for the validation set
        surv_df = model.predict_surv_df(X_val)

        # Compute c-index
        ev = EvalSurv(surv_df, T_val, E_val, censor_surv="km")
        c_index = ev.concordance_td()

        # Memory cleanup
        del model, net, ev, surv_df
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return c_index

    # Run study
    optuna.logging.set_verbosity(optuna.logging.WARNING) #
    study = optuna.create_study(direction="maximize")

    study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=True)

    return study.best_params

In [43]:
n_trials_ct = 20

# # Sim Linear
# best_linear_ct = optimise_coxtime(X_tr_lin, T_tr_lin, E_tr_lin, X_val_lin, T_val_lin, E_val_lin, n_trials=n_trials_ct)
# params_linear_ct = best_linear_ct.copy()

# # Sim Nonlinear Quadratic
# best_nonlinear_quad_ct = optimise_coxtime(X_tr_quad, T_tr_quad, E_tr_quad, X_val_quad, T_val_quad, E_val_quad, n_trials=n_trials_ct)
# params_nonlinear_quad_ct = best_nonlinear_quad_ct.copy()


# # gbsg
# best_gbsg_params_ct = optimise_coxtime(X_tr_gbsg, T_tr_gbsg, E_tr_gbsg, X_val_gbsg, T_val_gbsg, E_val_gbsg, n_trials=n_trials_ct)
# params_gbsg_ct = best_gbsg_params_ct.copy()

# # flchain
# best_fl_params_ct = optimise_coxtime(X_tr_fl, T_tr_fl, E_tr_fl, X_val_fl, T_val_fl, E_val_fl, n_trials=n_trials_ct)
# params_flchain_ct = best_fl_params_ct.copy()

# support
best_sup_params_ct = optimise_coxtime(X_tr_sup, T_tr_sup, E_tr_sup, X_val_sup, T_val_sup, E_val_sup, n_trials=n_trials_ct)
params_sup_ct = best_sup_params_ct.copy()

  0%|          | 0/20 [00:00<?, ?it/s]

In [44]:
# Print all optimal hyperparameters
print(f"params_linear_ct = {params_linear_ct}")
print(f"params_nonlinear_quad_ct = {params_nonlinear_quad_ct}")
print(f"params_gbsg_ct = {params_gbsg_ct}")
print(f"params_flchain_ct = {params_flchain_ct}")
print(f"params_sup_ct = {params_sup_ct}")

params_linear_ct = {'nodes': 192, 'layers': 2, 'dropout': 0.3326126419124013, 'lr': 0.00010307430742812953, 'batch_size': 128}
params_nonlinear_quad_ct = {'nodes': 96, 'layers': 2, 'dropout': 0.49259240223343587, 'lr': 0.008007012751455283, 'batch_size': 512}
params_gbsg_ct = {'nodes': 96, 'layers': 2, 'dropout': 0.24132002263793081, 'lr': 0.0007762201877873591, 'batch_size': 256}
params_flchain_ct = {'nodes': 96, 'layers': 1, 'dropout': 0.1104027705565437, 'lr': 0.0017097955858230902, 'batch_size': 64}
params_sup_ct = {'nodes': 160, 'layers': 2, 'dropout': 0.19753389340274102, 'lr': 0.0011697110879632367, 'batch_size': 512}


In [26]:
def evaluate_dataset(X_train, T_train, E_train,
                     X_test, T_test, E_test,
                     dataset_name, params_rsf,
                     params_ds, params_ct,
                     n_bootstraps=1000):

    # Scale features strictly on the training data to prevent leakage
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # format targets for C-index
    T_test = np.array(T_test)
    E_test = np.array(E_test)

    # format targets for IBS
    y_train_sksurv = np.array([(bool(e), t) for e, t in zip(E_train, T_train)], dtype=[('e', bool), ('t', float)])
    y_test_sksurv = np.array([(bool(e), t) for e, t in zip(E_test, T_test)], dtype=[('e', bool), ('t', float)])

    # define the time grid for IBS integration (5th to 95th percentile of test events)
    mask = E_test == 1
    event_times = T_test[mask]
    times = np.linspace(np.percentile(event_times, 5), np.percentile(event_times, 95), 100)

    # train models and get predictions
    risk_rsf, surv_rsf = rsf(X_train_scaled, T_train, E_train, X_test_scaled, params_rsf, times=times)
    risk_cph, surv_cph = cph(X_train_scaled, T_train, E_train, X_test_scaled, times=times)
    risk_lasso, surv_lasso = cox_lasso(X_train_scaled, T_train, E_train, X_test_scaled, times=times)
    risk_ds, surv_ds = deepsurv(X_train_scaled, T_train, E_train, X_test_scaled, params_ds, times=times)
    risk_ct, surv_ct = coxtime(X_train_scaled, T_train, E_train, X_test_scaled, params_ct, times=times)

    # Initialise lists for C-index
    cidx_cph, cidx_lasso, cidx_ds, cidx_rsf, cidx_ct = [], [], [], [], []
    # Initialise lists for IBS
    ibs_cph, ibs_lasso, ibs_ds, ibs_rsf, ibs_ct = [], [], [], [], []

    # Bootstrap loop
    for i in tqdm(range(n_bootstraps), desc=f"Bootstrapping {dataset_name}", leave=False):
        # Resample the test set indices with replacement
        indices = resample(np.arange(len(X_test_scaled)), replace=True, random_state=i)

        T_boot = T_test[indices]
        E_boot = E_test[indices]
        y_test_boot = y_test_sksurv[indices]

        if np.sum(E_boot) == 0:
            continue

        # c index
        cidx_cph.append(concordance_index(T_boot, -risk_cph[indices], E_boot) if risk_cph is not None else np.nan)
        cidx_lasso.append(concordance_index(T_boot, -risk_lasso[indices], E_boot) if risk_lasso is not None else np.nan)
        cidx_ds.append(concordance_index(T_boot, -risk_ds[indices], E_boot) if risk_ds is not None else np.nan)
        cidx_ct.append(concordance_index(T_boot, -risk_ct[indices], E_boot) if risk_ct is not None else np.nan)
        cidx_rsf.append(concordance_index(T_boot, -risk_rsf[indices], E_boot) if risk_rsf is not None else np.nan)

        time_mask = (times >= T_boot.min()) & (times < T_boot.max())
        times_boot = times[time_mask]

        if len(times_boot) < 2:
            continue

        # ibs score
        ibs_cph.append(integrated_brier_score(y_train_sksurv, y_test_boot, surv_cph[indices][:, time_mask], times_boot) if surv_cph is not None else np.nan)
        ibs_lasso.append(integrated_brier_score(y_train_sksurv, y_test_boot, surv_lasso[indices][:, time_mask], times_boot) if surv_lasso is not None else np.nan)
        ibs_ds.append(integrated_brier_score(y_train_sksurv, y_test_boot, surv_ds[indices][:, time_mask], times_boot) if surv_ds is not None else np.nan)
        ibs_ct.append(integrated_brier_score(y_train_sksurv, y_test_boot, surv_ct[indices][:, time_mask], times_boot) if surv_ct is not None else np.nan)
        ibs_rsf.append(integrated_brier_score(y_train_sksurv, y_test_boot, surv_rsf[indices][:, time_mask], times_boot) if surv_rsf is not None else np.nan)

    # Helper function to extract Mean and 95% CI
    def get_metrics(boot_list):
        if not boot_list:
            return np.nan, (np.nan, np.nan)
        return round(np.mean(boot_list), 3), (round(np.percentile(boot_list, 2.5), 3), round(np.percentile(boot_list, 97.5), 3))

    return {
        "Dataset": dataset_name,
        # C-Index Results
        "C-Index CPH (Mean)": get_metrics(cidx_cph)[0], "C-Index CPH (95% CI)": get_metrics(cidx_cph)[1],
        "C-Index Cox Lasso (Mean)": get_metrics(cidx_lasso)[0], "C-Index Cox Lasso (95% CI)": get_metrics(cidx_lasso)[1],
        "C-Index DeepSurv (Mean)": get_metrics(cidx_ds)[0], "C-Index DeepSurv (95% CI)": get_metrics(cidx_ds)[1],
        "C-Index CoxTime (Mean)": get_metrics(cidx_ct)[0], "C-Index CoxTime (95% CI)": get_metrics(cidx_ct)[1],
        "C-Index RSF (Mean)": get_metrics(cidx_rsf)[0], "C-Index RSF (95% CI)": get_metrics(cidx_rsf)[1],

        # IBS Results
        "IBS CPH (Mean)": get_metrics(ibs_cph)[0], "IBS CPH (95% CI)": get_metrics(ibs_cph)[1],
        "IBS Cox Lasso (Mean)": get_metrics(ibs_lasso)[0], "IBS Cox Lasso (95% CI)": get_metrics(ibs_lasso)[1],
        "IBS DeepSurv (Mean)": get_metrics(ibs_ds)[0], "IBS DeepSurv (95% CI)": get_metrics(ibs_ds)[1],
        "IBS CoxTime (Mean)": get_metrics(ibs_ct)[0], "IBS CoxTime (95% CI)": get_metrics(ibs_ct)[1],
        "IBS RSF (Mean)": get_metrics(ibs_rsf)[0], "IBS RSF (95% CI)": get_metrics(ibs_rsf)[1],
    }

In [ ]:
n_bootstraps = 1000
final_results = []

# Sim Linear
final_results.append(evaluate_dataset(
    X_train_lin, T_train_lin, E_train_lin,
    X_test_lin, T_test_lin, E_test_lin,
    "Sim Linear", params_linear_rsf, params_linear, params_linear_ct, n_bootstraps
))

# Sim Nonlinear Quadratic
final_results.append(evaluate_dataset(
    X_train_quad, T_train_quad, E_train_quad,
    X_test_quad, T_test_quad, E_test_quad,
    "Sim Nonlinear Quadratic", params_linear_rsf, params_nonlinear_quad, params_nonlinear_quad_ct, n_bootstraps
))

# GBSG
final_results.append(evaluate_dataset(
    X_train_gbsg, T_train_gbsg, E_train_gbsg,
    X_test_gbsg, T_test_gbsg, E_test_gbsg,
    "GBSG", params_linear_rsf, params_gbsg, params_gbsg_ct, n_bootstraps
))

# FLCHAIN
final_results.append(evaluate_dataset(
    X_train_fl, T_train_fl, E_train_fl,
    X_test_fl, T_test_fl, E_test_fl,
    "FLCHAIN", params_linear_rsf, params_flchain, params_flchain_ct, n_bootstraps
))

Bootstrapping Sim Linear:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrapping Sim Nonlinear Quadratic:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrapping GBSG:   0%|          | 0/1000 [00:00<?, ?it/s]

CPH failed to converge: Convergence halted due to matrix inversion problems. Suspicion is high collinearity. Please see the following tips in the lifelines documentation: https://lifelines.readthedocs.io/en/latest/Examples.html#problems-with-convergence-in-the-cox-proportional-hazard-modelMatrix is singular.


Bootstrapping FLCHAIN:   0%|          | 0/1000 [00:00<?, ?it/s]

,Dataset,C-Index CPH (Mean),C-Index CPH (95% CI),C-Index Cox Lasso (Mean),C-Index Cox Lasso (95% CI),C-Index DeepSurv (Mean),C-Index DeepSurv (95% CI),C-Index CoxTime (Mean),C-Index CoxTime (95% CI),C-Index RSF (Mean),C-Index RSF (95% CI)
0,Sim Linear,0.786,"(0.77, 0.802)",0.786,"(0.771, 0.803)",0.786,"(0.77, 0.802)",0.776,"(0.76, 0.792)",0.781,"(0.765, 0.797)"
1,Sim Nonlinear Quadratic,0.509,"(0.487, 0.532)",0.509,"(0.486, 0.532)",0.772,"(0.754, 0.789)",0.763,"(0.743, 0.781)",0.765,"(0.746, 0.782)"
2,GBSG,0.692,"(0.631, 0.755)",0.693,"(0.632, 0.756)",0.690,"(0.624, 0.75)",0.644,"(0.568, 0.723)",0.701,"(0.636, 0.76)"
3,FLCHAIN,NaN,"(nan, nan)",0.789,"(0.764, 0.813)",0.784,"(0.759, 0.808)",0.785,"(0.759, 0.809)",0.785,"(0.761, 0.809)"


,Dataset,IBS CPH (Mean),IBS CPH (95% CI),IBS Cox Lasso (Mean),IBS Cox Lasso (95% CI),IBS DeepSurv (Mean),IBS DeepSurv (95% CI),IBS CoxTime (Mean),IBS CoxTime (95% CI),IBS RSF (Mean),IBS RSF (95% CI)
0,Sim Linear,0.132,"(0.123, 0.142)",0.132,"(0.123, 0.142)",0.133,"(0.124, 0.142)",0.138,"(0.128, 0.148)",0.135,"(0.125, 0.146)"
1,Sim Nonlinear Quadratic,0.216,"(0.206, 0.225)",0.216,"(0.206, 0.225)",0.140,"(0.131, 0.149)",0.145,"(0.136, 0.155)",0.142,"(0.132, 0.152)"
2,GBSG,0.162,"(0.14, 0.187)",0.162,"(0.14, 0.187)",0.162,"(0.14, 0.185)",0.170,"(0.147, 0.196)",0.153,"(0.13, 0.178)"
3,FLCHAIN,NaN,"(nan, nan)",0.098,"(0.089, 0.107)",0.099,"(0.09, 0.108)",0.110,"(0.099, 0.122)",0.099,"(0.09, 0.109)"


In [45]:
# SUPPORT
final_results.append(evaluate_dataset(
    X_train_sup, T_train_sup, E_train_sup,
    X_test_sup, T_test_sup, E_test_sup,
    "SUPPORT", params_sup_rsf, params_sup, params_sup_ct, n_bootstraps
))

# Convert the massive list of dictionaries into a single DataFrame
df_all = pd.DataFrame(final_results)

# Isolate the C-Index columns
df_cindex = df_all[['Dataset']].join(df_all.filter(like='C-Index'))
display(df_cindex)

# Isolate the IBS columns
df_ibs = df_all[['Dataset']].join(df_all.filter(like='IBS'))
display(df_ibs)

Bootstrapping SUPPORT:   0%|          | 0/1000 [00:00<?, ?it/s]

,Dataset,C-Index CPH (Mean),C-Index CPH (95% CI),C-Index Cox Lasso (Mean),C-Index Cox Lasso (95% CI),C-Index DeepSurv (Mean),C-Index DeepSurv (95% CI),C-Index CoxTime (Mean),C-Index CoxTime (95% CI),C-Index RSF (Mean),C-Index RSF (95% CI)
0,Sim Linear,0.786,"(0.77, 0.802)",0.786,"(0.771, 0.803)",0.786,"(0.77, 0.802)",0.776,"(0.76, 0.792)",0.781,"(0.765, 0.797)"
1,Sim Nonlinear Quadratic,0.509,"(0.487, 0.532)",0.509,"(0.486, 0.532)",0.772,"(0.754, 0.789)",0.763,"(0.743, 0.781)",0.765,"(0.746, 0.782)"
2,GBSG,0.692,"(0.631, 0.755)",0.693,"(0.632, 0.756)",0.690,"(0.624, 0.75)",0.644,"(0.568, 0.723)",0.701,"(0.636, 0.76)"
3,FLCHAIN,NaN,"(nan, nan)",0.789,"(0.764, 0.813)",0.784,"(0.759, 0.808)",0.785,"(0.759, 0.809)",0.785,"(0.761, 0.809)"
4,SUPPORT,0.570,"(0.551, 0.587)",0.570,"(0.551, 0.586)",0.616,"(0.598, 0.632)",0.613,"(0.596, 0.63)",0.618,"(0.6, 0.636)"


,Dataset,IBS CPH (Mean),IBS CPH (95% CI),IBS Cox Lasso (Mean),IBS Cox Lasso (95% CI),IBS DeepSurv (Mean),IBS DeepSurv (95% CI),IBS CoxTime (Mean),IBS CoxTime (95% CI),IBS RSF (Mean),IBS RSF (95% CI)
0,Sim Linear,0.132,"(0.123, 0.142)",0.132,"(0.123, 0.142)",0.133,"(0.124, 0.142)",0.138,"(0.128, 0.148)",0.135,"(0.125, 0.146)"
1,Sim Nonlinear Quadratic,0.216,"(0.206, 0.225)",0.216,"(0.206, 0.225)",0.140,"(0.131, 0.149)",0.145,"(0.136, 0.155)",0.142,"(0.132, 0.152)"
2,GBSG,0.162,"(0.14, 0.187)",0.162,"(0.14, 0.187)",0.162,"(0.14, 0.185)",0.170,"(0.147, 0.196)",0.153,"(0.13, 0.178)"
3,FLCHAIN,NaN,"(nan, nan)",0.098,"(0.089, 0.107)",0.099,"(0.09, 0.108)",0.110,"(0.099, 0.122)",0.099,"(0.09, 0.109)"
4,SUPPORT,0.224,"(0.219, 0.23)",0.224,"(0.219, 0.23)",0.212,"(0.205, 0.218)",0.213,"(0.207, 0.22)",0.211,"(0.204, 0.218)"
